In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Creating Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("AmazonReviews_Task2") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.1.2


# Loading Daataset

In [3]:
df = spark.read.json(
    "amazon_reviews/raw/review_categories/Books.jsonl"
)

In [4]:
df.show(10, truncate=False)

+----------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Inspecting the dataset

In [5]:
df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [6]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 29475453
Columns: 10


In [7]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+----+------------+------+-----------+------+----+---------+-----+-------+-----------------+
|asin|helpful_vote|images|parent_asin|rating|text|timestamp|title|user_id|verified_purchase|
+----+------------+------+-----------+------+----+---------+-----+-------+-----------------+
|   0|           0|     0|          0|     0|   0|        0|    0|      0|                0|
+----+------------+------+-----------+------+----+---------+-----+-------+-----------------+



In [8]:
print(df.dtypes)

[('asin', 'string'), ('helpful_vote', 'bigint'), ('images', 'array<struct<attachment_type:string,large_image_url:string,medium_image_url:string,small_image_url:string>>'), ('parent_asin', 'string'), ('rating', 'double'), ('text', 'string'), ('timestamp', 'bigint'), ('title', 'string'), ('user_id', 'string'), ('verified_purchase', 'boolean')]


In [9]:
duplicate_count = df.count() - df.dropDuplicates().count()

print("Duplicate Rows :", duplicate_count)

Duplicate Rows : 333012


In [10]:
from pyspark.sql.functions import col, count, when, trim

df.select(
    count(when(col("asin").isNull() | (trim(col("asin")) == ""), "asin")).alias("asin"),
    count(when(col("parent_asin").isNull() | (trim(col("parent_asin")) == ""), "parent_asin")).alias("parent_asin"),
    count(when(col("text").isNull() | (trim(col("text")) == ""), "text")).alias("text"),
    count(when(col("title").isNull() | (trim(col("title")) == ""), "title")).alias("title"),
    count(when(col("user_id").isNull() | (trim(col("user_id")) == ""), "user_id")).alias("user_id"),
    count(when(col("rating").isNull(), "rating")).alias("rating"),
    count(when(col("helpful_vote").isNull(), "helpful_vote")).alias("helpful_vote"),
    count(when(col("timestamp").isNull(), "timestamp")).alias("timestamp"),
    count(when(col("verified_purchase").isNull(), "verified_purchase")).alias("verified_purchase"),
    count(when(col("images").isNull(), "images")).alias("images")
).show(truncate=False)

+----+-----------+----+-----+-------+------+------------+---------+-----------------+------+
|asin|parent_asin|text|title|user_id|rating|helpful_vote|timestamp|verified_purchase|images|
+----+-----------+----+-----+-------+------+------------+---------+-----------------+------+
|0   |0          |7636|0    |0      |0     |0           |0        |0                |0     |
+----+-----------+----+-----+-------+------+------------+---------+-----------------+------+



# Data preprocessing

In [11]:
rows_before = df.count()

df = df.dropDuplicates()

rows_after = df.count()

print("Rows Before :", rows_before)
print("Rows After  :", rows_after)
print("Duplicates Removed :", rows_before - rows_after)

Rows Before : 29475453
Rows After  : 29142441
Duplicates Removed : 333012


In [12]:
from pyspark.sql.functions import col, trim

df = df.filter(
    (col("text").isNotNull()) &
    (trim(col("text")) != "")
)

print("Rows After Removing Missing Text:", df.count())

Rows After Removing Missing Text: 29134934


# Limiting the data to 11 million

In [13]:
df = df.limit(11000000)

print("Final Rows:", df.count())

Final Rows: 11000000


In [14]:
import os

print(os.environ.get("HADOOP_HOME"))

C:\hadoop


In [15]:
df.write.mode("overwrite").parquet("amazon_reviews_clean_11M")

# Verifying the Reduced Dataset

In [16]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.show(5, truncate=False)

Rows: 11000000
Columns: 10
+----------+------------+------+-----------+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Partitioning the Dataset

In [3]:
df = spark.read.parquet("amazon_reviews_clean_11M")

In [4]:
print("Current Partitions:", df.rdd.getNumPartitions())

Current Partitions: 29


In [5]:
df = df.coalesce(8)

print("Partitions:", df.rdd.getNumPartitions())

Partitions: 8


# Feature Engineering

In [6]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "label",
    when(col("rating") >= 4, 1).otherwise(0)
)

df.select("rating", "label").show(10)

+------+-----+
|rating|label|
+------+-----+
|   4.0|    1|
|   5.0|    1|
|   4.0|    1|
|   5.0|    1|
|   5.0|    1|
|   5.0|    1|
|   5.0|    1|
|   5.0|    1|
|   5.0|    1|
|   5.0|    1|
+------+-----+
only showing top 10 rows


In [7]:
df = df.withColumn(
    "verified_purchase",
    col("verified_purchase").cast("integer")
)

df.select("verified_purchase").show(10)

+-----------------+
|verified_purchase|
+-----------------+
|                0|
|                0|
|                0|
|                0|
|                0|
|                0|
|                0|
|                0|
|                0|
|                0|
+-----------------+
only showing top 10 rows


In [8]:
feature_df = df.select(
    "helpful_vote",
    "verified_purchase",
    "label"
)

feature_df.show(10)

+------------+-----------------+-----+
|helpful_vote|verified_purchase|label|
+------------+-----------------+-----+
|           0|                0|    1|
|           0|                0|    1|
|          17|                0|    1|
|           2|                0|    1|
|           0|                0|    1|
|           0|                0|    1|
|           2|                0|    1|
|           0|                0|    1|
|           0|                0|    1|
|           0|                0|    1|
+------------+-----------------+-----+
only showing top 10 rows


In [9]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "helpful_vote",
        "verified_purchase"
    ],
    outputCol="features"
)

feature_df = assembler.transform(feature_df)

feature_df.select(
    "features",
    "label"
).show(10, truncate=False)

+----------+-----+
|features  |label|
+----------+-----+
|[0.0,0.0] |1    |
|[0.0,0.0] |1    |
|[17.0,0.0]|1    |
|[2.0,0.0] |1    |
|[0.0,0.0] |1    |
|[0.0,0.0] |1    |
|[2.0,0.0] |1    |
|[0.0,0.0] |1    |
|[0.0,0.0] |1    |
|[0.0,0.0] |1    |
+----------+-----+
only showing top 10 rows


In [10]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=False
)

scalerModel = scaler.fit(feature_df)

feature_df = scalerModel.transform(feature_df)

feature_df.select(
    "scaledFeatures",
    "label"
).show(10, truncate=False)

+-------------------------+-----+
|scaledFeatures           |label|
+-------------------------+-----+
|[0.0,0.0]                |1    |
|[0.0,0.0]                |1    |
|[1.0113166342035778,0.0] |1    |
|[0.11897842755336209,0.0]|1    |
|[0.0,0.0]                |1    |
|[0.0,0.0]                |1    |
|[0.11897842755336209,0.0]|1    |
|[0.0,0.0]                |1    |
|[0.0,0.0]                |1    |
|[0.0,0.0]                |1    |
+-------------------------+-----+
only showing top 10 rows


In [11]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=[
        assembler,
        scaler
    ]
)

print("Pipeline Created Successfully")

Pipeline Created Successfully


In [16]:
print(feature_df.columns)

['helpful_vote', 'verified_purchase', 'label', 'features', 'scaledFeatures']


In [19]:
feature_df.select(
    "features",
    "scaledFeatures",
    "label"
).show(10, truncate=False)

+----------+-------------------------+-----+
|features  |scaledFeatures           |label|
+----------+-------------------------+-----+
|[0.0,0.0] |[0.0,0.0]                |1    |
|[0.0,0.0] |[0.0,0.0]                |1    |
|[17.0,0.0]|[1.0113166342035778,0.0] |1    |
|[2.0,0.0] |[0.11897842755336209,0.0]|1    |
|[0.0,0.0] |[0.0,0.0]                |1    |
|[0.0,0.0] |[0.0,0.0]                |1    |
|[2.0,0.0] |[0.11897842755336209,0.0]|1    |
|[0.0,0.0] |[0.0,0.0]                |1    |
|[0.0,0.0] |[0.0,0.0]                |1    |
|[0.0,0.0] |[0.0,0.0]                |1    |
+----------+-------------------------+-----+
only showing top 10 rows


In [22]:
train_df, test_df = feature_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training Rows:", train_df.count())
print("Testing Rows:", test_df.count())

Training Rows: 8799451
Testing Rows: 2200549


In [23]:
print("Train Columns:", train_df.columns)
print("Test Columns:", test_df.columns)

Train Columns: ['helpful_vote', 'verified_purchase', 'label', 'features', 'scaledFeatures']
Test Columns: ['helpful_vote', 'verified_purchase', 'label', 'features', 'scaledFeatures']


# Saving Training and Testing Dataset

In [24]:
train_df.write \
    .mode("overwrite") \
    .parquet("train_df")

In [25]:
test_df.write \
    .mode("overwrite") \
    .parquet("test_df")

In [26]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'amazon_reviews', 'amazon_reviews_clean_11M', 'amazon_reviews_clean_11M.zip', 'Task1.ipynb', 'Task2.ipynb', 'Task3.ipynb', 'test_df', 'train_df']
